<a href="https://www.kaggle.com/code/adegbaju/fifa-world-cup-2026-group-stage-match-prediction?scriptVersionId=330651111" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# FIFA World Cup 2026 – Group Stage Match Prediction
====================================================

Dataset: world_cup_2026_group_stage_results.xlsx

This script:
 - Loads the match results
 - Builds team‑level performance features
 - Trains a Random Forest classifier to predict match outcome (H/D/A)
 - Evaluates the model
 - Predicts a completely new fixture (e.g., a knockout or future group match)


DO NOT FORGET TO UPVOTE

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# LOAD & INSPECT DATA

In [2]:
FILE_PATH = "/kaggle/input/datasets/adegbaju/fifa-world-cup-2026-group-stage/world_cup_2026_group_stage_results.xlsx"
df = pd.read_excel(FILE_PATH, engine='openpyxl')

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Sample:")
print(df.head(), "\n")

# Basic sanity checks
assert {'home_team', 'away_team', 'home_score', 'away_score', 'match_outcome'}.issubset(df.columns), \
       "Required columns missing!"

Dataset shape: (50, 8)
Columns: ['group', 'home_team', 'away_team', 'home_score', 'away_score', 'goal_difference', 'total_goals', 'match_outcome']
Sample:
     group    home_team     away_team  home_score  away_score  \
0  Group A       Mexico  South Africa           2           0   
1  Group A  South Korea       Czechia           2           1   
2  Group A      Czechia  South Africa           1           1   
3  Group A       Mexico   South Korea           1           0   
4  Group A      Czechia        Mexico           0           2   

   goal_difference  total_goals match_outcome  
0                2            2             H  
1                1            3             H  
2                0            2             D  
3                1            1             H  
4               -2            2             A   



# FEATURE ENGINEERING 

In [3]:
# Make a copy to avoid modifying original
matches = df.copy()

## Home / Away split for team stats

In [4]:
home = matches[['home_team', 'home_score', 'away_score', 'match_outcome']].copy()
home.columns = ['team', 'goals_for', 'goals_against', 'outcome']
home['home'] = 1

away = matches[['away_team', 'away_score', 'home_score', 'match_outcome']].copy()
away.columns = ['team', 'goals_for', 'goals_against', 'outcome']
away['home'] = 0
# For away matches, outcome needs to be flipped: H->A, A->H, D->D
away['outcome'] = away['outcome'].map({'H':'A', 'A':'H', 'D':'D'})

all_records = pd.concat([home, away], ignore_index=True)

## Compute aggregated team statistics

In [5]:
team_stats = all_records.groupby('team').agg(
    games_played = ('goals_for', 'count'),
    avg_goals_scored = ('goals_for', 'mean'),
    avg_goals_conceded = ('goals_against', 'mean'),
    # win/draw/loss rates
    win_rate = ('outcome', lambda x: (x == 'H').mean()),   # winning from the team's perspective
    draw_rate = ('outcome', lambda x: (x == 'D').mean()),
    loss_rate = ('outcome', lambda x: (x == 'A').mean()),
    # home/away split
    home_avg_goals_scored = ('goals_for', lambda x: x[all_records.loc[x.index, 'home'] == 1].mean()),
    home_avg_goals_conceded = ('goals_against', lambda x: x[all_records.loc[x.index, 'home'] == 1].mean()),
    away_avg_goals_scored = ('goals_for', lambda x: x[all_records.loc[x.index, 'home'] == 0].mean()),
    away_avg_goals_conceded = ('goals_against', lambda x: x[all_records.loc[x.index, 'home'] == 0].mean()),
).reset_index()

# Fill NaN values that occur when a team has not played any home/away match yet
# with the overall tournament averages.
overall_avg = {
    'avg_goals_scored': all_records['goals_for'].mean(),
    'avg_goals_conceded': all_records['goals_against'].mean(),
    'win_rate': all_records['outcome'].eq('H').mean(),
    'draw_rate': all_records['outcome'].eq('D').mean(),
    'loss_rate': all_records['outcome'].eq('A').mean(),
    'home_avg_goals_scored': home['goals_for'].mean(),
    'home_avg_goals_conceded': home['goals_against'].mean(),
    'away_avg_goals_scored': away['goals_for'].mean(),
    'away_avg_goals_conceded': away['goals_against'].mean(),
}

for col in ['avg_goals_scored','avg_goals_conceded','win_rate','draw_rate','loss_rate',
            'home_avg_goals_scored','home_avg_goals_conceded',
            'away_avg_goals_scored','away_avg_goals_conceded']:
    team_stats[col] = team_stats[col].fillna(overall_avg[col])

print("Team stats table (first 5):")
print(team_stats.head(), "\n")

Team stats table (first 5):
        team  games_played  avg_goals_scored  avg_goals_conceded  win_rate  \
0    Algeria             2               1.0                 2.0       0.5   
1  Argentina             2               2.5                 0.0       1.0   
2  Australia             2               1.0                 1.0       0.5   
3    Austria             2               1.5                 1.5       0.5   
4    Belgium             2               0.5                 0.5       0.0   

   draw_rate  loss_rate  home_avg_goals_scored  home_avg_goals_conceded  \
0        0.0        0.5                   1.92                     1.04   
1        0.0        0.0                   2.50                     0.00   
2        0.0        0.5                   2.00                     0.00   
3        0.0        0.5                   3.00                     1.00   
4        1.0        0.0                   0.50                     0.50   

   away_avg_goals_scored  away_avg_goals_conceded  



## Build match‑level features

In [6]:
def create_match_features(row, team_stats):
    """
    For a given match row (home_team, away_team), construct a feature vector
    using the aggregated team statistics.
    """
    home_team = row['home_team']
    away_team = row['away_team']

    home_stats = team_stats[team_stats['team'] == home_team].squeeze()
    away_stats = team_stats[team_stats['team'] == away_team].squeeze()

    # If a team is completely new (not in stats), use overall averages
    if home_stats.empty:
        home_stats = pd.Series(overall_avg, name=home_team)
    if away_stats.empty:
        away_stats = pd.Series(overall_avg, name=away_team)

    features = {
        # Attacking strength
        'home_avg_goals_scored': home_stats['avg_goals_scored'],
        'away_avg_goals_scored': away_stats['avg_goals_scored'],
        # Defensive strength
        'home_avg_goals_conceded': home_stats['avg_goals_conceded'],
        'away_avg_goals_conceded': away_stats['avg_goals_conceded'],
        # Home/away specific performance
        'home_home_goals_scored': home_stats.get('home_avg_goals_scored', overall_avg['home_avg_goals_scored']),
        'away_away_goals_scored': away_stats.get('away_avg_goals_scored', overall_avg['away_avg_goals_scored']),
        # Form indicators
        'home_win_rate': home_stats['win_rate'],
        'away_win_rate': away_stats['win_rate'],
        'home_draw_rate': home_stats['draw_rate'],
        'away_draw_rate': away_stats['draw_rate'],
        # Recent form can be added (e.g., last 3 matches) but not enough data here.
        # Home advantage flag
        'is_home': 1   # always 1 for home team in this dataset, but we keep it
    }
    return pd.Series(features)

# Apply to every match in the dataset to create training data
X = matches.apply(lambda row: create_match_features(row, team_stats), axis=1)
y = matches['match_outcome']   # H, D, A

print("Feature matrix shape:", X.shape)
print("First 3 rows of features:")
print(X.head(3), "\n")


Feature matrix shape: (50, 11)
First 3 rows of features:
   home_avg_goals_scored  away_avg_goals_scored  home_avg_goals_conceded  \
0               1.666667               0.666667                 0.000000   
1               0.666667               0.666667                 1.000000   
2               0.666667               0.666667                 1.666667   

   away_avg_goals_conceded  home_home_goals_scored  away_away_goals_scored  \
0                 1.000000                     1.5                     0.5   
1                 1.666667                     2.0                     1.0   
2                 1.000000                     0.5                     0.5   

   home_win_rate  away_win_rate  home_draw_rate  away_draw_rate  is_home  
0       1.000000       0.333333        0.000000        0.333333      1.0  
1       0.333333       0.000000        0.000000        0.333333      1.0  
2       0.000000       0.333333        0.333333        0.333333      1.0   



# LABEL ENCODER

In [7]:
# Encode target labels: 0=H, 1=A, 2=D
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

Label mapping: {'A': 0, 'D': 1, 'H': 2}


# TRAIN / TEST SPLIT

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTraining set size: {len(X_train)}, Test set size: {len(X_test)}")



Training set size: 40, Test set size: 10


# MODEL TRAINING

In [9]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight='balanced'  # because draw is usually rare
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=5, n_estimators=200,
                       random_state=42)

# MODEL EVALUATION

In [10]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {acc:.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Cross‑validation on full dataset to get a realistic picture
cv_scores = cross_val_score(model, X, y_encoded, cv=5, scoring='accuracy')
print(f"5‑fold CV Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

# Feature importance
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
print("\nTop 5 important features:")
print(feature_importance.sort_values(ascending=False).head(5))


Test Accuracy: 1.00

Classification Report:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00         2
           D       1.00      1.00      1.00         3
           H       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

5‑fold CV Accuracy: 0.84 (+/- 0.05)

Top 5 important features:
away_draw_rate             0.138819
home_home_goals_scored     0.124932
home_draw_rate             0.122208
home_win_rate              0.114873
away_avg_goals_conceded    0.109302
dtype: float64


# PREDICT A NEXT MATCH

In [11]:
# Example (a fixture not in the dataset)
next_match = {
    'home_team': 'Uruguay',
    'away_team': 'Spain'
}
next_features = create_match_features(pd.Series(next_match), team_stats)
next_features_df = pd.DataFrame([next_features])  # model expects 2D array

proba = model.predict_proba(next_features_df)
pred_class = model.predict(next_features_df)
pred_label = le.inverse_transform(pred_class)[0]

print(f"\n{'='*50}")
print(f"Next match prediction: {next_match['home_team']} vs {next_match['away_team']}")
print(f"Predicted outcome: {pred_label}")
print("Outcome probabilities:")
for label, prob in zip(le.classes_, proba[0]):
    print(f"  {label} : {prob:.2%}")

# Optional: display the feature values used for prediction
print("\nFeature vector for this fixture:")
print(next_features.to_string())


Next match prediction: Uruguay vs Spain
Predicted outcome: A
Outcome probabilities:
  A : 45.42%
  D : 30.86%
  H : 23.73%

Feature vector for this fixture:
home_avg_goals_scored      2.00
away_avg_goals_scored      4.00
home_avg_goals_conceded    2.00
away_avg_goals_conceded    0.00
home_home_goals_scored     2.00
away_away_goals_scored     1.04
home_win_rate              0.00
away_win_rate              1.00
home_draw_rate             1.00
away_draw_rate             0.00
is_home                    1.00


# **Knockout Stage Simulation & Champion Prediction**
=====================================================================
Uses the group-stage match results (world_cup_2026_group_stage_results.xlsx)
to compute real group standings and the 32 advancing teams.
Then trains a Random Forest classifier on those group matches and
simulates the knockout phase 1000 times to estimate each team's
championship probability.

# BUILD TEAM STATS

In [12]:
# Split each match into home & away records to calculate per‑team metrics
home = df[['home_team', 'home_score', 'away_score', 'match_outcome']].copy()
home.columns = ['team', 'goals_for', 'goals_against', 'outcome']
home['is_home'] = 1

away = df[['away_team', 'away_score', 'home_score', 'match_outcome']].copy()
away.columns = ['team', 'goals_for', 'goals_against', 'outcome']
away['is_home'] = 0
# Flip outcome for away team's perspective
away['outcome'] = away['outcome'].map({'H': 'A', 'A': 'H', 'D': 'D'})

all_records = pd.concat([home, away], ignore_index=True)

# Aggregate team statistics
team_stats = all_records.groupby('team').agg(
    games=('goals_for', 'count'),
    avg_goals_scored=('goals_for', 'mean'),
    avg_goals_conceded=('goals_against', 'mean'),
    win_rate=('outcome', lambda x: (x == 'H').mean()),
    draw_rate=('outcome', lambda x: (x == 'D').mean()),
    loss_rate=('outcome', lambda x: (x == 'A').mean()),
    home_goals_scored=('goals_for', lambda x: x[all_records.loc[x.index, 'is_home'] == 1].mean()),
    away_goals_scored=('goals_for', lambda x: x[all_records.loc[x.index, 'is_home'] == 0].mean()),
    home_goals_conceded=('goals_against', lambda x: x[all_records.loc[x.index, 'is_home'] == 1].mean()),
    away_goals_conceded=('goals_against', lambda x: x[all_records.loc[x.index, 'is_home'] == 0].mean()),
).reset_index()

# Fill NaN (teams that played no home/away matches) with overall averages
overall_avg = {
    'avg_goals_scored': all_records['goals_for'].mean(),
    'avg_goals_conceded': all_records['goals_against'].mean(),
    'win_rate': all_records['outcome'].eq('H').mean(),
    'draw_rate': all_records['outcome'].eq('D').mean(),
    'loss_rate': all_records['outcome'].eq('A').mean(),
    'home_goals_scored': home['goals_for'].mean(),
    'away_goals_scored': away['goals_for'].mean(),
    'home_goals_conceded': home['goals_against'].mean(),
    'away_goals_conceded': away['goals_against'].mean(),
}
for col in overall_avg:
    team_stats[col] = team_stats[col].fillna(overall_avg[col])

# FEATURE ENGINEERING

In [13]:
def match_features(row, team_stats, neutral=False):
    """
    Build feature vector for a match.
    If neutral=True, treat both teams as 'away' (no home advantage) and
    use overall averages rather than home/away split.
    """
    home_team = row['home_team']
    away_team = row['away_team']

    home_stats = team_stats[team_stats['team'] == home_team]
    away_stats = team_stats[team_stats['team'] == away_team]

    # If team not in stats (should not happen), use overall averages
    if home_stats.empty:
        home_stats = pd.DataFrame([overall_avg])
    if away_stats.empty:
        away_stats = pd.DataFrame([overall_avg])
    home_stats = home_stats.squeeze()
    away_stats = away_stats.squeeze()

    if neutral:
        # No home advantage – use overall averages for both sides
        feat = {
            'home_avg_goals_scored': home_stats['avg_goals_scored'],
            'away_avg_goals_scored': away_stats['avg_goals_scored'],
            'home_avg_goals_conceded': home_stats['avg_goals_conceded'],
            'away_avg_goals_conceded': away_stats['avg_goals_conceded'],
            'home_home_goals': home_stats['avg_goals_scored'],  # neutral -> treat as general strength
            'away_away_goals': away_stats['avg_goals_scored'],
            'home_win_rate': home_stats['win_rate'],
            'away_win_rate': away_stats['win_rate'],
            'home_draw_rate': home_stats['draw_rate'],
            'away_draw_rate': away_stats['draw_rate'],
            'is_home': 0
        }
    else:
        feat = {
            'home_avg_goals_scored': home_stats['avg_goals_scored'],
            'away_avg_goals_scored': away_stats['avg_goals_scored'],
            'home_avg_goals_conceded': home_stats['avg_goals_conceded'],
            'away_avg_goals_conceded': away_stats['avg_goals_conceded'],
            'home_home_goals': home_stats['home_goals_scored'],
            'away_away_goals': away_stats['away_goals_scored'],
            'home_win_rate': home_stats['win_rate'],
            'away_win_rate': away_stats['win_rate'],
            'home_draw_rate': home_stats['draw_rate'],
            'away_draw_rate': away_stats['draw_rate'],
            'is_home': 1
        }
    return pd.Series(feat)

# Build training data (actual group stage matches, with home advantage)
X = df.apply(lambda row: match_features(row, team_stats, neutral=False), axis=1)
y = df['match_outcome']
le = LabelEncoder()
y_enc = le.fit_transform(y)   # 0=H, 1=A, 2=D

# TRAIN CLASSIFIER

In [14]:
clf = RandomForestClassifier(n_estimators=200, max_depth=5,
                             random_state=42, class_weight='balanced')
clf.fit(X, y_enc)
print("Classifier trained. Label order:", le.classes_)

Classifier trained. Label order: ['A' 'D' 'H']


# GROUP STANDINGS 

In [15]:
def compute_standings(matches_df):
    """
    Compute final group tables from the given match records.
    Returns a dict: group -> DataFrame with columns [Team, Pts, GF, GA, GD].
    Tie-breakers: points > GD > GF > head‑to‑head (simplified: direct points, then GD in h2h)
    """
    groups = {}
    for group in matches_df['group'].unique():
        grp = matches_df[matches_df['group'] == group]
        teams = set(grp['home_team']).union(set(grp['away_team']))
        table = pd.DataFrame({'Team': list(teams)})
        table['Pts'] = 0
        table['GF'] = 0
        table['GA'] = 0
        table.set_index('Team', inplace=True)

        for _, row in grp.iterrows():
            ht = row['home_team']
            at = row['away_team']
            gf_h = row['home_score']
            gf_a = row['away_score']
            table.at[ht, 'GF'] += gf_h
            table.at[ht, 'GA'] += gf_a
            table.at[at, 'GF'] += gf_a
            table.at[at, 'GA'] += gf_h

            if gf_h > gf_a:
                table.at[ht, 'Pts'] += 3
            elif gf_h < gf_a:
                table.at[at, 'Pts'] += 3
            else:
                table.at[ht, 'Pts'] += 1
                table.at[at, 'Pts'] += 1

        table['GD'] = table['GF'] - table['GA']
        table.reset_index(inplace=True)

        # Sort by points, GD, GF, then head‑to‑head
        table = table.sort_values(['Pts', 'GD', 'GF', 'Team'], ascending=[False, False, False, True])
        # Head‑to‑head simplification: only break ties manually for top positions
        # (For a realistic simulation we rely on the sorting above; it's sufficient.)
        groups[group] = table
    return groups

group_tables = compute_standings(df)
print("\nGroup standings computed.")
for g, tab in group_tables.items():
    print(f"\nGroup {g}:")
    print(tab[['Team', 'Pts', 'GD', 'GF']].to_string(index=False))


Group standings computed.

Group Group A:
        Team  Pts  GD  GF
      Mexico    9   5   5
South Africa    4  -1   2
 South Korea    3  -1   2
     Czechia    1  -3   2

Group Group B:
          Team  Pts  GD  GF
Bosnia & Herz.    4   2   4
   Switzerland    4   1   3
        Canada    1  -1   2
         Qatar    1  -2   2

Group Group C:
    Team  Pts  GD  GF
  Brazil    4   3   4
 Morocco    4   1   2
Scotland    3   0   1
   Haiti    0  -4   0

Group Group D:
         Team  Pts  GD  GF
United States    6   5   6
    Australia    3   0   2
     Paraguay    3  -2   2
     TÃ¼rkiye    0  -3   0

Group Group E:
       Team  Pts  GD  GF
    Germany    6   7   9
Ivory Coast    3   0   2
    Ecuador    1  -1   0
   CuraÃ§ao    1  -6   1

Group Group F:
       Team  Pts  GD  GF
Netherlands    4   4   7
      Japan    4   4   6
     Sweden    3   0   6
    Tunisia    0  -8   1

Group Group G:
       Team  Pts  GD  GF
      Egypt    4   2   4
    IR Iran    2   0   2
    Belgium    2   0 

# ADVANCING TEAMS

In [16]:
# Top 2 from each group
group_winners = []
group_runners_up = []
third_placed = []

for g in ['Group A','Group B','Group C','Group D','Group E','Group F',
          'Group G','Group H','Group I','Group J','Group K','Group L']:
    tab = group_tables[g]
    group_winners.append(tab.iloc[0]['Team'])
    group_runners_up.append(tab.iloc[1]['Team'])
    third_placed.append({
        'Team': tab.iloc[2]['Team'],
        'Group': g,
        'Pts': tab.iloc[2]['Pts'],
        'GD': tab.iloc[2]['GD'],
        'GF': tab.iloc[2]['GF']
    })

# Sort third‑placed teams and pick top 8
third_placed = pd.DataFrame(third_placed)
third_placed = third_placed.sort_values(['Pts','GD','GF'], ascending=[False,False,False])
best_thirds = third_placed.head(8)['Team'].tolist()
print("\nBest third‑placed teams advancing:", best_thirds)

# Store the 32 advancing teams
advancing = group_winners + group_runners_up + best_thirds
assert len(advancing) == 32, "Expected 32 teams"


Best third‑placed teams advancing: ['Senegal', 'Sweden', 'Scotland', 'Croatia', 'South Korea', 'Paraguay', 'Algeria', 'Belgium']


# KNOCKOUT BRACKET MAPPING


Round of 32 fixture definitions (official 2026 bracket)
Format: (team1_slot, team2_slot) where slot is e.g. '1A'=winner group A,
'2A'=runner-up group A, '3best1'=best 3rd #1, etc.
We assign best 3rds to specific matches according to FIFA rules.
I'll hardcode the known 2026 R32 schedule.
Group winners are seeded 1A–1L.
The bracket uses 8 best 3rd place teams that are placed based on a table
(not random). I'll implement a simplified but correct mapping.

In [17]:
# First, rank best 3rds 1 to 8 based on points, GD, GF.
best_thirds_ranked = third_placed.head(8)['Team'].tolist()

# Official match schedule (1A vs 3C/D/E/F means the best 3rd from groups C,D,E,F).
# I'll use a mapping from real rules:
# Match 73: 1A vs 3C/D/E/F (best 3rd from those groups, chosen automatically)
# Match 74: 1C vs 2F
# Match 75: 1B vs 3E/F/G/H
# Match 76: 1D vs 2E
# Match 77: 1E vs 2A
# Match 78: 1G vs 2B
# Match 79: 1F vs 2C
# Match 80: 1H vs 2D
# Match 81: 1I vs 3A/B/C/D  (wait, need to be precise)
# Actually, the full bracket is complex. For simplicity, I will pair:
#   - all group winners vs 3rd place teams or runners‑up such that
#     no group winner faces a team from its own group, and 3rd places are
#     spread according to a fixed pattern.
# I'll use a pre‑defined dict of R32 matches based on standard 48-team format.
# (source: FIFA.com/World Cup 2026 match schedule)
# Match order:
r32_matchups = [
    ('1A', '3C'),   # winner A vs 3rd C (best 3rd from C) – simplified
    ('1C', '2F'),
    ('1B', '3E'),   # etc.
    ('1D', '2E'),
    ('1E', '2A'),
    ('1G', '2B'),
    ('1F', '2C'),
    ('1H', '2D'),
    ('1I', '3A'),
    ('1K', '2J'),
    ('1J', '2K'),
    ('1L', '2I'),
    # 4 more matches: runners‑up vs 3rd place
    ('2L', '3B'),
    ('2G', '3D'),
    ('2H', '3F'),
    ('2J', '3H'),
]
# This is not fully accurate but provides a deterministic bracket.
# In a production setting you would map exactly.
# Fill the slots.
def get_team(slot):
    """Return the team name for a slot like '1A', '2B', '3C'."""
    g_letter = slot[-1]  # e.g., 'A'
    group_name = f'Group {g_letter}'
    if slot.startswith('1'):
        idx = ord(g_letter) - ord('A')
        return group_winners[idx]
    elif slot.startswith('2'):
        idx = ord(g_letter) - ord('A')
        return group_runners_up[idx]
    elif slot.startswith('3'):
        # 3rd place from that specific group (not best overall) – matches actual rule
        idx = ord(g_letter) - ord('A')
        return group_tables[group_name].iloc[2]['Team']
    else:
        raise ValueError(f'Invalid slot: {slot}')

# Build round of 32 matches as list of (home, away) – but knockout is neutral,
# we'll treat first team as "home" in the classifier just for prediction,
# then ignore home advantage by setting neutral=True.
r32_fixtures = [(get_team(a), get_team(b)) for a, b in r32_matchups]

print("\nRound of 32 fixtures (home‑listed first):")
for i, (h, a) in enumerate(r32_fixtures, 1):
    print(f"  {i}: {h} vs {a}")


Round of 32 fixtures (home‑listed first):
  1: Mexico vs Scotland
  2: Brazil vs Japan
  3: Bosnia & Herz. vs Ecuador
  4: United States vs Ivory Coast
  5: Germany vs South Africa
  6: Egypt vs Switzerland
  7: Netherlands vs Morocco
  8: Spain vs Australia
  9: France vs South Korea
  10: Colombia vs Austria
  11: Argentina vs Portugal
  12: England vs Norway
  13: Ghana vs Canada
  14: IR Iran vs Paraguay
  15: Cape Verde vs Sweden
  16: Austria vs Uruguay


# KNOCKOUT SIMULATOR

In [18]:
def simulate_knockout_match(team1, team2, model, le, team_stats):
    """
    Predict outcome of a neutral knockout match.
    Returns the winning team name.
    """
    # Create feature row (neutral venue)
    match = pd.Series({'home_team': team1, 'away_team': team2})
    feat = match_features(match, team_stats, neutral=True).to_frame().T

    proba = model.predict_proba(feat)[0]   # [P(H), P(A), P(D)]
    # Map probabilities
    prob_dict = {le.inverse_transform([i])[0]: proba[i] for i in range(3)}
    # If draw, we go to extra time / penalties – resolve with a coin flip
    # but weighted by overall win rate (or just 50/50). Here we use a simple
    # random draw breaker: compare overall win_rate stats.
    if prob_dict.get('D', 0) > max(prob_dict.get('H', 0), prob_dict.get('A', 0)):
        # Resolve draw by team strength
        t1_stats = team_stats[team_stats['team'] == team1]
        t2_stats = team_stats[team_stats['team'] == team2]
        wr1 = t1_stats['win_rate'].values[0] if not t1_stats.empty else 0.33
        wr2 = t2_stats['win_rate'].values[0] if not t2_stats.empty else 0.33
        # Weighted random: team with higher win rate more likely to win
        p1 = wr1 / (wr1 + wr2) if (wr1 + wr2) > 0 else 0.5
        return team1 if np.random.rand() < p1 else team2
    else:
        # Regular outcome
        if prob_dict['H'] > prob_dict['A']:
            return team1
        else:
            return team2

def simulate_tournament(r32_fixtures, model, le, team_stats):
    """
    Simulate full knockout phase: R32 → R16 → QF → SF → Final.
    Returns the champion.
    """
    # R32
    r32_winners = []
    for h, a in r32_fixtures:
        winner = simulate_knockout_match(h, a, model, le, team_stats)
        r32_winners.append(winner)

    # R16 (8 matches) – bracket ordering: adjacent pairs
    r16_winners = []
    for i in range(0, 16, 2):
        w = simulate_knockout_match(r32_winners[i], r32_winners[i+1], model, le, team_stats)
        r16_winners.append(w)

    # QF (4 matches)
    qf_winners = []
    for i in range(0, 8, 2):
        w = simulate_knockout_match(r16_winners[i], r16_winners[i+1], model, le, team_stats)
        qf_winners.append(w)

    # SF (2 matches)
    sf_winners = []
    w1 = simulate_knockout_match(qf_winners[0], qf_winners[1], model, le, team_stats)
    w2 = simulate_knockout_match(qf_winners[2], qf_winners[3], model, le, team_stats)
    sf_winners = [w1, w2]

    # Final
    champion = simulate_knockout_match(sf_winners[0], sf_winners[1], model, le, team_stats)
    return champion

# MONTE CARLO SIMULATION

In [19]:
N_SIM = 1000
champions = []
print(f"\nRunning {N_SIM} simulations...")
for sim in range(N_SIM):
    champ = simulate_tournament(r32_fixtures, clf, le, team_stats)
    champions.append(champ)
    if (sim+1) % 200 == 0:
        print(f"  {sim+1} simulations completed")

# Count frequencies
champ_counts = pd.Series(champions).value_counts()
champ_probs = (champ_counts / N_SIM).sort_values(ascending=False)

print("\n\n=========== CHAMPIONSHIP PROBABILITIES ===========")
print(f"{'Team':<20} {'Win Probability':>15}")
print("-" * 40)
for team, prob in champ_probs.head(10).items():
    print(f"{team:<20} {prob:>14.1%}")

print("\nMost likely champion:", champ_probs.index[0], f"({champ_probs.iloc[0]:.1%})")


Running 1000 simulations...
  200 simulations completed
  400 simulations completed
  600 simulations completed
  800 simulations completed
  1000 simulations completed


=========== CHAMPIONSHIP PROBABILITIES ===========
Team                 Win Probability
----------------------------------------
France                       100.0%

Most likely champion: France (100.0%)
